# Sales by Customer - Aggregation Analysis

This notebook processes the sales data from `input.csv` to create a customer-level summary.

## Task
- Group orders by customer
- Calculate total amount spent per customer
- Count number of orders per customer

## Validation Plan
- Conservation check: total amounts before and after aggregation must match
- Column structure verification
- Null value check

In [ ]:
import pandas as pd

# Read the input sales data
df = pd.read_csv('input.csv')
print("Input data shape:", df.shape)
print("\nFirst few rows:")
df.head()

## Data Aggregation

Grouping by customer and computing:
- Sum of amounts per customer
- Count of orders per customer

In [ ]:
# Perform the aggregation
result = df.groupby('customer').agg(
    total_amount=('amount', 'sum'),
    order_count=('order_id', 'count')
).reset_index()

# Sort by customer name
result = result.sort_values('customer').reset_index(drop=True)

print("Aggregated result:")
result

## Validation

Running embedded validation checks:

In [ ]:
# Validation 1: Conservation check - total amounts must match
original_total = df['amount'].sum()
aggregated_total = result['total_amount'].sum()
assert abs(original_total - aggregated_total) < 0.01, \
    f"Conservation check failed: {original_total} != {aggregated_total}"

# Validation 2: Column structure
expected_columns = {'customer', 'total_amount', 'order_count'}
actual_columns = set(result.columns)
assert expected_columns == actual_columns, \
    f"Column mismatch: expected {expected_columns}, got {actual_columns}"

# Validation 3: No null values (compute once, reuse in the message)
null_counts = result.isnull().sum()
assert null_counts.sum() == 0, f"Null values detected: {null_counts.to_dict()}"

print("All validation checks passed!")
print(f"- Conservation: ${original_total:.2f} = ${aggregated_total:.2f}")
print(f"- Columns: {list(result.columns)}")
print(f"- Null values: 0")

In [ ]:
# Save the result to its own output file. expected-output.csv is the reference
# oracle — compare against it, never overwrite it.
result.to_csv('sales-by-customer.csv', index=False)

expected = pd.read_csv('expected-output.csv')
pd.testing.assert_frame_equal(
    result.reset_index(drop=True), expected.reset_index(drop=True), check_dtype=False
)
print("Result saved to sales-by-customer.csv and matches expected-output.csv")

# Display final result
result